# PostgreSQL Streaming Replication

Un espacio para **jugar** con todo lo del proyecto: SQL, índices, transacciones, la replicación en vivo,
`synchronous_commit`, el WAL, la decodificación lógica y tus propias tablas. Las lecciones (`lessons/`) muestran cada
tema en orden; aquí puedes cambiar cualquier cosa y volver a ejecutar.

**Cómo usarlo**

1. Ejecuta las celdas de **Configuración** (secciones 0 y 1) una vez.
2. Después salta a la sección que quieras. Las celdas marcadas con **Juega aquí** están hechas para que las
   cambies: consultas, índices, niveles de aislamiento, cargas de escritura.
3. Nada de lo que hagas aquí rompe el proyecto: las secciones 6, 8 y 10 usan sus propias tablas (`sandbox_*`), y
   si alteras `employees`, `python lessons/00_seed.py` la deja como al principio en segundos.

> Requisito: haber ejecutado `python lessons/00_seed.py` al menos una vez.

| # | Sección | Qué puedes probar |
| --- | --- | --- |
| 0 | Configuración | Conexión, versión y topología de los nodos |
| 1 | Herramientas | Funciones `sql`, `en_cada_nodo`, `plan`, `topologia`, `replicacion`, `wal_generado` |
| 2 | Explorar la tabla | `SELECT`, estructura de la tabla, cualquier consulta |
| 3 | Índices y planes | `EXPLAIN ANALYZE` con y sin índice, índices parciales y de expresión |
| 4 | Agregaciones y ventanas | `GROUP BY`, funciones de ventana y gráficos |
| 5 | Transacciones y MVCC | Niveles de aislamiento, versiones de fila (`xmin`, `xmax`), `VACUUM` |
| 6 | Replicación en vivo | Lag de los standby durante una carga masiva |
| 7 | Consistencia | `synchronous_commit`, latencia y lecturas desactualizadas |
| 8 | WAL | Cuánto WAL genera cada operación; tablas `UNLOGGED` |
| 9 | Decodificación lógica | Tus cambios como eventos con un slot lógico |
| 10 | Tu propia tabla | Tu esquema, `jsonb`, índices GIN y lectura desde un standby |

## 0. Configuración

La conexión sale de las variables `PG*` de `.envs/postgres.env`: `PGHOST=pg1,pg2,pg3` y
`PGTARGETSESSIONATTRS=read-write`, así que `connect()` siempre llega al primary, aunque haya habido un failover. El
notebook usa las mismas funciones que las lecciones (`src/lab/`).

In [ ]:
import threading
import time

import matplotlib.pyplot as plt
import pandas as pd
import psycopg
from psycopg import sql as pgsql

from lab.client import connect, connect_node, node_name, standbys, topology, wait_for_standbys
from lab.config import NODES

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

conn = connect()
PRIMARY = node_name(conn)
STANDBYS = standbys(conn)

print("Versión:   ", conn.execute("SHOW server_version").fetchone()["server_version"])
print("Primary:   ", PRIMARY, "| standby:", STANDBYS)
print("Base:      ", conn.info.dbname, "| tablas:",
      [r["tablename"] for r in conn.execute("SELECT tablename FROM pg_tables WHERE schemaname = 'public' ORDER BY 1")])
print("Empleados: ", conn.execute("SELECT count(*) AS n FROM employees").fetchone()["n"])
pd.DataFrame(topology())

## 1. Herramientas

Funciones cortas para no repetir código. Están aquí, a la vista, para que las leas y las cambies.

| Función | Qué hace |
| --- | --- |
| `sql(consulta, params, nodo)` | Ejecuta SQL y devuelve un DataFrame. Sin `nodo` va al primary; con `nodo="pg2"`, a ese nodo |
| `en_cada_nodo(consulta)` | La misma consulta directamente en cada nodo, en una sola tabla |
| `plan(consulta)` | Imprime el plan de ejecución real (`EXPLAIN ANALYZE`) |
| `topologia()` | Rol de cada nodo: primary, standby, bloqueado o caído |
| `replicacion()` | `pg_stat_replication`: estado y retraso de cada standby |
| `wal_generado(consulta)` | Ejecuta la consulta y devuelve cuántos bytes de WAL produjo |

In [ ]:
def sql(consulta, params=None, nodo=None):
    """Ejecuta SQL. Devuelve un DataFrame si hay filas, o el mensaje del servidor (p. ej. 'UPDATE 3')."""
    with (connect_node(nodo) if nodo else connect()) as c:
        cur = c.execute(consulta, params)
        return pd.DataFrame(cur.fetchall()) if cur.description else cur.statusmessage


def en_cada_nodo(consulta, params=None):
    """La misma consulta en cada nodo, conectándose directamente a cada uno."""
    partes = []
    for nodo in NODES:
        try:
            df = sql(consulta, params, nodo=nodo)
            df.insert(0, "nodo", nodo)
        except psycopg.OperationalError:
            df = pd.DataFrame([{"nodo": nodo, "error": "caído"}])
        partes.append(df)
    return pd.concat(partes, ignore_index=True)


def plan(consulta, params=None):
    for fila in conn.execute(f"EXPLAIN (ANALYZE, BUFFERS, COSTS OFF, TIMING OFF, SUMMARY OFF) {consulta}", params):
        print(fila["QUERY PLAN"])


def topologia():
    return pd.DataFrame(topology())


def replicacion():
    return sql("""
        SELECT application_name AS standby, state, sync_state, sent_lsn, replay_lsn,
               pg_wal_lsn_diff(pg_current_wal_lsn(), replay_lsn)::bigint AS lag_bytes, replay_lag
        FROM pg_stat_replication ORDER BY application_name
    """)


def wal_generado(consulta, params=None):
    """Bytes de WAL entre antes y después de ejecutar la consulta (incluye lo que escriban otras sesiones)."""
    with connect() as c:
        inicio = c.execute("SELECT pg_current_wal_insert_lsn() AS lsn").fetchone()["lsn"]
        c.execute(consulta, params)
        return c.execute("SELECT pg_wal_lsn_diff(pg_current_wal_insert_lsn(), %s)::bigint AS b",
                         (inicio, )).fetchone()["b"]


replicacion()

## 2. Explorar la tabla

SQL normal contra el primary. Las lecturas también pueden ir a un standby: `sql(..., nodo="pg2")`.

In [ ]:
# Estructura de la tabla (lo que muestra \d employees en psql)
sql("""
    SELECT column_name, data_type, is_nullable, column_default
    FROM information_schema.columns WHERE table_name = 'employees' ORDER BY ordinal_position
""")

In [ ]:
# Una fila completa, leída desde un standby
sql("SELECT * FROM employees WHERE employee_id = %s", ("E0001", ), nodo=STANDBYS[0])

### Juega aquí: cualquier consulta

In [ ]:
sql("""
    SELECT name, department, city, salary, skills
    FROM employees
    WHERE department = 'Ingeniería' AND 'python' = ANY(skills)
    ORDER BY salary DESC
    LIMIT 5
""")

# Ideas:
#   WHERE name ILIKE '%ramírez%'                               búsqueda por patrón
#   WHERE skills @> ARRAY['aws', 'docker']                     contiene todos
#   WHERE hired_at >= now() - interval '6 months'
#   SELECT city, count(*) FROM employees GROUP BY city

## 3. Índices y planes

`EXPLAIN ANALYZE` ejecuta la consulta y muestra el plan real: qué recorrió, cuántas filas descartó y cuántas
páginas leyó (`Buffers`). Con pocas filas el planificador a veces prefiere `Seq Scan` aunque exista un índice:
leer 30 páginas seguidas puede salir más barato que saltar entre el índice y la tabla.

In [ ]:
# Índices de la tabla, su tamaño y cuántas veces se han usado
sql("""
    SELECT indexrelname AS indice, pg_size_pretty(pg_relation_size(indexrelid)) AS tamano, idx_scan AS usos
    FROM pg_stat_user_indexes WHERE relname = 'employees' ORDER BY indexrelname
""")

### Juega aquí: crea un índice y compara el plan

In [ ]:
consulta = "SELECT name, salary FROM employees WHERE city = 'Cali' AND salary > 15000000"

sql("DROP INDEX IF EXISTS sandbox_city_salary_idx")
print("--- sin índice")
plan(consulta)

sql("CREATE INDEX sandbox_city_salary_idx ON employees (city, salary)")   # prueba (salary, city), o solo (city)
sql("ANALYZE employees")
print("\n--- con índice")
plan(consulta)

# Otros tipos de índice para probar:
#   CREATE INDEX ... ON employees (salary) WHERE active                 parcial: solo filas activas
#   CREATE INDEX ... ON employees (lower(name))                         de expresión: WHERE lower(name) = '...'
#   CREATE INDEX ... ON employees USING brin (hired_at)                 BRIN: diminuto, para datos ordenados
#   SET enable_seqscan = off  (en la misma conexión)                    obliga a usar índices, para comparar

In [ ]:
# Limpieza: este índice no lo usa ninguna lección
sql("DROP INDEX IF EXISTS sandbox_city_salary_idx")

## 4. Agregaciones y ventanas

In [ ]:
salarios = sql("SELECT department, salary FROM employees WHERE active")
orden = salarios.groupby("department")["salary"].median().sort_values().index

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot([salarios[salarios.department == d]["salary"] / 1e6 for d in orden], tick_labels=orden, orientation="horizontal")
ax.set_xlabel("salario (millones)")
ax.set_title("Salario por departamento (empleados activos)")
plt.tight_layout()

### Juega aquí: funciones de ventana

In [ ]:
sql("""
    SELECT employee_id, name, department, hired_at::date AS hired_at, salary,
           row_number() OVER w                              AS orden_ingreso,
           salary - lag(salary) OVER w                      AS vs_anterior,
           round(avg(salary) OVER (PARTITION BY department)) AS promedio_depto
    FROM employees
    WHERE department = 'Finanzas'
    WINDOW w AS (PARTITION BY department ORDER BY hired_at)
    ORDER BY hired_at
    LIMIT 10
""")

# Ideas:
#   ntile(4) OVER (ORDER BY salary)                         cuartil salarial
#   sum(salary) OVER (ORDER BY hired_at)                    nómina acumulada en el tiempo
#   percent_rank() OVER (PARTITION BY city ORDER BY salary)

## 5. Transacciones y MVCC

PostgreSQL no sobrescribe filas: cada `UPDATE` crea una **versión nueva** y marca la anterior como muerta. Cada
transacción ve las versiones que corresponden a su *snapshot*. `xmin` es la transacción que creó la versión,
`xmax` la que la borró o reemplazó, y `ctid` su posición física (página, fila).

In [ ]:
# Versiones de una fila: el UPDATE cambia ctid (nueva posición) y xmin (nueva transacción)
antes = sql("SELECT ctid, xmin, xmax, salary FROM employees WHERE employee_id = 'E0005'")
sql("UPDATE employees SET salary = salary + 1 WHERE employee_id = 'E0005'")
despues = sql("SELECT ctid, xmin, xmax, salary FROM employees WHERE employee_id = 'E0005'")
pd.concat([antes.assign(momento="antes"), despues.assign(momento="después")])

In [ ]:
# Las versiones muertas ocupan espacio hasta que VACUUM las limpia (autovacuum lo hace solo, cada tanto).
# Todo en la misma conexión: cada sesión publica sus estadísticas cada tanto; aquí se fuerza para verlas ya.
def tuplas():
    conn.execute("SELECT pg_stat_force_next_flush()")
    conn.execute("SELECT pg_stat_clear_snapshot()")
    return pd.DataFrame(conn.execute("""
        SELECT n_live_tup AS vivas, n_dead_tup AS muertas, last_vacuum,
               pg_size_pretty(pg_relation_size('employees')) AS tamano
        FROM pg_stat_user_tables WHERE relname = 'employees'
    """).fetchall())


conn.execute("UPDATE employees SET salary = salary + 1 WHERE department = 'Ventas'")
display(tuplas())
conn.execute("VACUUM employees")
display(tuplas())

`VACUUM` limpió las versiones muertas de antes, pero no las de este último `UPDATE`. La razón es la
replicación: con `hot_standby_feedback = on`, cada standby le informa al primary, por su slot, la transacción más
antigua que aún puede necesitar (`xmin`). El primary no borra versiones que un standby podría estar leyendo, y así
las consultas largas en los standby no se cancelan. El standby actualiza ese dato cada ~10 s
(`wal_receiver_status_interval`).

In [ ]:
display(sql("SELECT slot_name, xmin FROM pg_replication_slots WHERE slot_type = 'physical'"))
time.sleep(11)
display(sql("SELECT slot_name, xmin FROM pg_replication_slots WHERE slot_type = 'physical'"))
conn.execute("VACUUM employees")
tuplas()

### Juega aquí: niveles de aislamiento

Dos sesiones a la vez. La sesión **B** lee un salario dos veces; entre ambas lecturas, la sesión **A** lo cambia y
hace COMMIT. Con `READ COMMITTED` (el valor por defecto) B ve el cambio en su segunda lectura; con
`REPEATABLE READ` sigue viendo su snapshot hasta que termina su transacción.

In [ ]:
def aislamiento(nivel):
    a, b = connect(autocommit=False), connect(autocommit=False)
    leer = lambda: b.execute("SELECT salary FROM employees WHERE employee_id = 'E0006'").fetchone()["salary"]
    try:
        b.execute(pgsql.SQL("SET TRANSACTION ISOLATION LEVEL {}").format(pgsql.SQL(nivel)))
        pasos = [("B lee", leer())]
        a.execute("UPDATE employees SET salary = salary + 100000 WHERE employee_id = 'E0006'")
        a.commit()
        pasos.append(("A suma 100.000 y hace COMMIT; B lee", leer()))
        b.commit()
        pasos.append(("B termina su transacción y vuelve a leer", leer()))
        b.commit()
    finally:
        a.close()
        b.close()
    return pd.DataFrame(pasos, columns=["paso", f"salario visto por B ({nivel})"])


display(aislamiento("READ COMMITTED"))
display(aislamiento("REPEATABLE READ"))   # prueba también "SERIALIZABLE"

## 6. Replicación en vivo

Mientras un hilo hace una carga masiva en el primary, otro mide cada 20 ms cuántos bytes de WAL le faltan aplicar
a cada standby. El retraso sube durante la carga y vuelve a 0 apenas termina.

### Juega aquí: el tamaño de la carga

In [ ]:
FILAS = 300_000          # prueba 50_000 o 1_000_000

sql("DROP TABLE IF EXISTS sandbox_load")
sql("CREATE TABLE sandbox_load (id bigint, payload text)")
muestras, fin = [], threading.Event()


def medir():
    with connect() as c:
        t0 = time.perf_counter()
        while not fin.is_set() or len(muestras) < 5 or any(m["lag_bytes"] > 0 for m in muestras[-len(STANDBYS):]):
            for r in c.execute("""
                SELECT application_name AS standby, pg_wal_lsn_diff(pg_current_wal_lsn(), replay_lsn)::bigint AS lag_bytes
                FROM pg_stat_replication
            """):
                muestras.append({"t": time.perf_counter() - t0, **r})
            time.sleep(0.02)
            if time.perf_counter() - t0 > 30:
                break


hilo = threading.Thread(target=medir)
hilo.start()
time.sleep(0.2)
inicio = time.perf_counter()
sql("INSERT INTO sandbox_load SELECT g, md5(g::text) FROM generate_series(1, %s) AS g", (FILAS, ))
print(f"Carga de {FILAS:,} filas: {time.perf_counter() - inicio:.2f} s")
fin.set()
hilo.join()

df = pd.DataFrame(muestras)
fig, ax = plt.subplots(figsize=(9, 3.5))
for standby, grupo in df.groupby("standby"):
    ax.plot(grupo["t"], grupo["lag_bytes"] / 1024 / 1024, label=standby)
ax.set_xlabel("segundos")
ax.set_ylabel("WAL pendiente (MB)")
ax.set_title("Retraso de los standby durante la carga")
ax.legend()
plt.tight_layout()

In [ ]:
# La tabla llegó entera a cada standby
wait_for_standbys(conn)
display(en_cada_nodo("SELECT count(*) AS filas, pg_size_pretty(pg_relation_size('sandbox_load')) AS tamano FROM sandbox_load"))
sql("DROP TABLE sandbox_load")

## 7. Consistencia

`synchronous_standby_names` define qué standby deben confirmar cada COMMIT; `synchronous_commit` hasta dónde se
espera. La celda deja la replicación **asíncrona** al terminar, aunque falle a mitad de camino: con standby
síncronos configurados y caídos, las escrituras quedarían bloqueadas.

### Juega aquí: elige la configuración síncrona

In [ ]:
SINCRONOS = f"ANY 1 ({', '.join(NODES)})"     # prueba f"ANY {len(STANDBYS)} (...)", "FIRST 1 (pg3, pg2)" o ""
NIVELES = ["off", "local", "remote_write", "on", "remote_apply"]
ESCRITURAS = 200

sql("CREATE TABLE IF NOT EXISTS consistency_check (id int PRIMARY KEY, counter int)")
sql("INSERT INTO consistency_check VALUES (1, 0) ON CONFLICT DO NOTHING")
filas = []
try:
    conn.execute(pgsql.SQL("ALTER SYSTEM SET synchronous_standby_names = {}").format(pgsql.Literal(SINCRONOS)))
    conn.execute("SELECT pg_reload_conf()")
    time.sleep(0.5)
    display(sql("SELECT application_name AS standby, sync_state FROM pg_stat_replication ORDER BY 1"))
    with connect() as w, connect_node(STANDBYS[0]) as r:
        for nivel in NIVELES:
            w.execute(pgsql.SQL("SET synchronous_commit = {}").format(pgsql.Identifier(nivel)))
            atrasadas, inicio = 0, time.perf_counter()
            for i in range(ESCRITURAS):
                w.execute("UPDATE consistency_check SET counter = %s WHERE id = 1", (i, ))
                atrasadas += r.execute("SELECT counter FROM consistency_check").fetchone()["counter"] != i
            filas.append({"synchronous_commit": nivel, "ms por commit": (time.perf_counter() - inicio) * 1000 / ESCRITURAS,
                          f"lecturas atrasadas en {STANDBYS[0]}": atrasadas})
finally:
    conn.execute("ALTER SYSTEM RESET synchronous_standby_names")
    conn.execute("SELECT pg_reload_conf()")

resultado = pd.DataFrame(filas)
display(resultado)
resultado.plot.bar(x="synchronous_commit", y="ms por commit", legend=False, figsize=(7, 3),
                   title=f"Latencia por commit con '{SINCRONOS}'", rot=0)
plt.tight_layout()

## 8. WAL

Todo lo que cambia en una tabla normal pasa por el WAL, y el WAL es lo que viaja a los standby. Aquí se mide
cuánto genera cada operación. Las tablas `UNLOGGED` no escriben WAL: son mucho más rápidas, pero se vacían si el
servidor se cae y **no se replican**.

In [ ]:
sql("DROP TABLE IF EXISTS sandbox_wal, sandbox_wal_idx, sandbox_unlogged")
sql("CREATE TABLE sandbox_wal (id int PRIMARY KEY, valor int, nota text)")
sql("CREATE TABLE sandbox_wal_idx (id int PRIMARY KEY, valor int, nota text)")
sql("CREATE INDEX ON sandbox_wal_idx (valor)")
sql("CREATE UNLOGGED TABLE sandbox_unlogged (id int PRIMARY KEY, valor int, nota text)")
sql("CHECKPOINT")   # tras un checkpoint, la 1.ª modificación de cada página escribe la página entera (full page write)

N = 10_000
medidas = {
    "INSERT (tabla con PK)": "INSERT INTO sandbox_wal SELECT g, g, 'x' FROM generate_series(1, %s) g",
    "INSERT (PK + índice en valor)": "INSERT INTO sandbox_wal_idx SELECT g, g, 'x' FROM generate_series(1, %s) g",
    "INSERT (UNLOGGED)": "INSERT INTO sandbox_unlogged SELECT g, g, 'x' FROM generate_series(1, %s) g",
    "UPDATE columna sin índice": "UPDATE sandbox_wal SET valor = valor + 1 WHERE id <= %s",
    "UPDATE columna con índice": "UPDATE sandbox_wal_idx SET valor = valor + 1 WHERE id <= %s",
    "DELETE": "DELETE FROM sandbox_wal WHERE id <= %s",
}
wal = pd.DataFrame([{"operación": k, "WAL (kB)": wal_generado(q, (N, )) / 1024} for k, q in medidas.items()])
display(wal.round(1))
wal.plot.barh(x="operación", y="WAL (kB)", legend=False, figsize=(8, 3), title=f"WAL generado por {N:,} filas")
plt.tight_layout()

In [ ]:
# La tabla UNLOGGED existe en los standby, pero no se puede leer: sus datos nunca viajaron
wait_for_standbys(conn)
for tabla in ("sandbox_wal_idx", "sandbox_unlogged"):
    try:
        print(tabla, "en", STANDBYS[0], "->", sql(f"SELECT count(*) AS n FROM {tabla}", nodo=STANDBYS[0])["n"][0], "filas")
    except psycopg.Error as error:
        print(tabla, "en", STANDBYS[0], "->", type(error).__name__ + ":", error.diag.message_primary)

### Juega aquí: mide tu propia operación

In [ ]:
print(wal_generado("UPDATE sandbox_wal_idx SET nota = repeat('y', 200) WHERE id <= 1000"), "bytes")

# Ideas:
#   "TRUNCATE sandbox_wal_idx"                               casi nada: no registra fila por fila
#   "DELETE FROM sandbox_wal_idx"                            compárala con TRUNCATE
#   "CREATE INDEX ON sandbox_wal_idx (id, valor)"
#   "VACUUM FULL sandbox_wal_idx"                            reescribe la tabla entera

In [ ]:
sql("DROP TABLE IF EXISTS sandbox_wal, sandbox_wal_idx, sandbox_unlogged")

## 9. Decodificación lógica

Un slot lógico traduce el WAL a cambios fila por fila (lección 07). Aquí puedes hacer cualquier cambio y ver
cómo queda como evento. Al final, **borra el slot**: mientras exista, el primary guarda todo el WAL que el slot no
haya consumido.

In [ ]:
SLOT = "sandbox_slot"
sql("SELECT pg_drop_replication_slot(slot_name) FROM pg_replication_slots WHERE slot_name = %s", (SLOT, ))
sql("SELECT * FROM pg_create_logical_replication_slot(%s, 'test_decoding')", (SLOT, ))


def cambios(consumir=False):
    funcion = "pg_logical_slot_get_changes" if consumir else "pg_logical_slot_peek_changes"
    return sql(f"SELECT lsn, xid, data FROM {funcion}(%s, NULL, NULL, 'skip-empty-xacts', '1')", (SLOT, ))

### Juega aquí: haz cambios y míralos como eventos

In [ ]:
with connect(autocommit=False) as c:
    c.execute("UPDATE employees SET city = 'Cali' WHERE employee_id = 'E0010'")
    c.execute("UPDATE employees SET skills = array_append(skills, 'rust') WHERE employee_id = 'E0011'")
    c.commit()
with connect(autocommit=False) as c:
    c.execute("DELETE FROM employees WHERE employee_id = 'E0012'")
    c.rollback()     # un ROLLBACK no genera eventos

cambios()            # cambios(consumir=True) los consume: la siguiente lectura empieza después

In [ ]:
# Limpieza: borrar el slot
sql("SELECT pg_drop_replication_slot(%s)", (SLOT, ))
print("slot", SLOT, "borrado")

## 10. Tu propia tabla

Una tabla `sandbox_items` con **tu** esquema y tus datos. Usa `jsonb` para guardar atributos libres (similar a
un documento de MongoDB), con índice GIN para buscar dentro. Todo se replica a los standby.

### Juega aquí: tu esquema y tus datos

In [ ]:
sql("DROP TABLE IF EXISTS sandbox_items")
sql("""
    CREATE TABLE sandbox_items (
        id        int GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        categoria text NOT NULL,
        nombre    text NOT NULL,
        atributos jsonb NOT NULL DEFAULT '{}'
    )
""")
mis_items = [
    ("energía", "Panel solar 550 W", {"potencia_w": 550, "tecnologia": "monocristalino", "garantia_anios": 25}),
    ("energía", "Inversor híbrido 5 kW", {"potencia_w": 5000, "fases": 1, "baterias": True}),
    ("energía", "Medidor bidireccional", {"fases": 3, "protocolos": ["DLMS", "Modbus"]}),
    ("cocina", "Cafetera de goteo", {"tazas": 12, "potencia_w": 900}),
    ("deporte", "Bicicleta de ruta", {"talla": "M", "material": "carbono", "cambios": 22}),
]
with connect() as c, c.cursor() as cur:
    cur.executemany("INSERT INTO sandbox_items (categoria, nombre, atributos) VALUES (%s, %s, %s)",
                    [(cat, nombre, psycopg.types.json.Jsonb(attrs)) for cat, nombre, attrs in mis_items])
sql("CREATE INDEX ON sandbox_items USING gin (atributos)")
sql("SELECT * FROM sandbox_items")

In [ ]:
# Consultas dentro de jsonb:  ->> extrae como texto, @> "contiene", ? "tiene la clave"
display(sql("SELECT nombre, (atributos->>'potencia_w')::int AS potencia_w FROM sandbox_items "
            "WHERE atributos ? 'potencia_w' ORDER BY potencia_w DESC"))
display(sql("SELECT nombre FROM sandbox_items WHERE atributos @> %s",
            (psycopg.types.json.Jsonb({"baterias": True}), )))

# Y desde un standby, en cuanto se replica
wait_for_standbys(conn)
sql("SELECT categoria, count(*) AS items FROM sandbox_items GROUP BY categoria", nodo=STANDBYS[-1])

In [ ]:
# Limpieza (opcional)
# sql("DROP TABLE sandbox_items")

---

**¿Rompiste algo?**

- `python lessons/00_seed.py` reconstruye `employees` y sus índices.
- Si las escrituras quedan bloqueadas (esperando standby síncronos): `ALTER SYSTEM RESET synchronous_standby_names;
  SELECT pg_reload_conf();`
- Si quedó un slot sin borrar: `SELECT pg_drop_replication_slot('sandbox_slot');`
- Para volver a la topología inicial: `docker compose -f docker-compose.yml -f replication.yml down -v` y `up` de nuevo.